In [3]:
import numpy as np
import pandas as pd
import sys

#sbatch --export=All -t 1:00:00 --mem=10G -o run_log/mergS.err -o run_log/mergS.out -J mergS --wrap="python merged_samples.py TM6_1000bp0ovl_filtered TM4_1000bp0ovl M3_1000bp0ovl"



def accessData(list_files, path, list_labels):
    """opens each file and stores in dictionary with key a label"""
    #create empty dictionary to store the dataframes
    dic_dataframes = {}
    #iterate to get the file names
    #make a counter to access the correct label
    count = 0
    for file in list_files:
        #open each file
        data_frame = pd.read_csv(path+file, sep='\t', index_col=[0,1,2,3])
        #add the dataframe as an element of a dictionary with key the filename
        dic_dataframes[list_labels[count]] = data_frame
        count = count + 1
    return dic_dataframes


def normalizeTo(dic_data, factor):
    """Normalize to median
    1. divide by the total reads in each column
    2. multiply by normalization factor"""
    #make a dictionary to store the normalized datasets
    dic_normal = {}
    #Get each dataframe
    for key in dic_data:
        #get the total transcript counts per section
        data_frame_total = dic_data[key].sum(axis='index')
        #divide each section by the total of the section
        data_frame_normal_s1 = dic_data[key].divide(data_frame_total, axis='columns')
        #multiply by the meadian value
        data_frame_normal = data_frame_normal_s1.multiply(factor)
        #save the dataframe to the dictionary
        dic_normal[key] = data_frame_normal
    return dic_normal

def saveCSV(dic_data, path, label):
    """Save dataframes inside dictionary to csv files"""
    for key in dic_data:
        dic_data[key].to_csv(path+key+label+'.bed', sep='\t', header=True)
    return print('Files saved')

# Create the merged count table that is not working on shark

In [2]:
#access de data
#Create a variable list with the file names and one with the labels to use as dictionary keys
path=''
file_names = ['TM6_1000bp0ovl_cpg_coverage_table.bed', 'TM4_1000bp0ovl_cpg_coverage_table.bed', 'M3_1000bp0ovl_cpg_coverage_table.bed']
labels = ['M1', 'M2', 'M3']

#use accessData() function to obtain a dictonary with each dataset with labels as key
data = accessData(file_names, path, labels)
data['M1']

/var/folders/mg/6wjwkry15y79vlj0f727ljh40000gq/T/ipykernel_13675/1045862020.py:36: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frame = pd.read_csv(path+file, sep='\t', index_col=[0,1,2,3])
/var/folders/mg/6wjwkry15y79vlj0f727ljh40000gq/T/ipykernel_13675/1045862020.py:36: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frame = pd.read_csv(path+file, sep='\t', index_col=[0,1,2,3])
/var/folders/mg/6wjwkry15y79vlj0f727ljh40000gq/T/ipykernel_13675/1045862020.py:36: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frame = pd.read_csv(path+file, sep='\t', index_col=[0,1,2,3])


s10  s11  s12  s13  s24  s25  s26  \
chr               start   end     Lpnp1                                      
1                 3000826 3001825 4        0    0   22    0    0    0    0   
                  3002826 3003825 5        6    0    0    0   12    0    0   
                  3003826 3004825 2        0    0    4   18    0    4    0   
                  3005826 3006825 1        0    0    0    0    0    0    0   
                  3006826 3007825 5        0    0    7    0    0    0    0   
...                                      ...  ...  ...  ...  ...  ...  ...   
Y_JH584300_random 138923  139922  5        0    0    0    0    0    0    0   
                  139923  140922  5        0    0    0    0    0    0    0   
                  140923  141922  2        0    0    0    0    0    0    0   
                  141923  142922  17       0    0    0    0    0    0    0   
                  142923  143922  5        0    0    0    0    0    0    0   

                                         s27  s35  s36  s37  s38  
chr               start   end     Lpnp1                           
1                 3000826 3001825 4        0    0    0    0    0  
                  3002826 3003825 5        0   11    0    0    0  
                  3003826 3004825 2        0    0    4    0    0  
                  3005826 3006825 1        0    0    0    0    0  
                  3006826 3007825 5        0    0    0    0    0  
...                                      ...  ...  ...  ...  ...  
Y_JH584300_random 138923  139922  5        0    0    0    0    0  
                  139923  140922  5        0    0    0    0    0  
                  140923  141922  2        0    0    0    0    0  
                  141923  142922  17       0    0    0    0    0  
                  142923  143922  5        0    0    0    0    0  

[2213351 rows x 12 columns]

In [4]:
print("M1 length: " + str(len(data['M1'].index)))
print("M2 length: " + str(len(data['M2'].index)))
print("M3 length: " + str(len(data['M3'].index)))
print("M1 M2 not equal " + str(sum(data['M1'].index != data['M2'].index)))
print("M1 M3 not equal " + str(sum(data['M1'].index != data['M3'].index)))

M1 length: 2213351
M2 length: 2213351
M3 length: 2213351
M1 M2 not equal 0
M1 M3 not equal 0


In [5]:
all_data={}
for id in labels:
    data[id] = data[id].astype(float)
    all_data[id] =data[id].add_prefix(id+'_')
    print(len(all_data[id].index))
all_data[id] 

2213351
2213351
2213351


M3_s1  M3_s2  M3_s3  M3_s4  M3_s17  \
chr               start   end     Lpnp1                                       
1                 3000826 3001825 4        0.0    0.0    0.0    4.0     0.0   
                  3002826 3003825 5       17.0    2.0    4.0    4.0    25.0   
                  3003826 3004825 2       10.0   56.0   44.0   22.0    20.0   
                  3005826 3006825 1        0.0    0.0    0.0    0.0     0.0   
                  3006826 3007825 5        1.0    0.0    1.0    2.0     2.0   
...                                        ...    ...    ...    ...     ...   
Y_JH584300_random 138923  139922  5        0.0    0.0    0.0    0.0     0.0   
                  139923  140922  5        0.0    0.0    0.0    0.0     0.0   
                  140923  141922  2        0.0    0.0    0.0    0.0     0.0   
                  141923  142922  17       0.0    0.0    0.0    0.0     0.0   
                  142923  143922  5        0.0    0.0    0.0    0.0     0.0   

                                         M3_s18  M3_s19  M3_s20  M3_s33  \
chr               start   end     Lpnp1                                   
1                 3000826 3001825 4         0.0     0.0     0.0     0.0   
                  3002826 3003825 5         8.0    19.0     5.0     8.0   
                  3003826 3004825 2        32.0    20.0    12.0    28.0   
                  3005826 3006825 1         0.0     0.0     0.0     0.0   
                  3006826 3007825 5         7.0     2.0     0.0     0.0   
...                                         ...     ...     ...     ...   
Y_JH584300_random 138923  139922  5         0.0     0.0     0.0     0.0   
                  139923  140922  5         0.0     0.0     0.0     0.0   
                  140923  141922  2         0.0     0.0     0.0     0.0   
                  141923  142922  17        0.0     0.0     0.0     0.0   
                  142923  143922  5         0.0     0.0     0.0     0.0   

                                         M3_s34  M3_s35  M3_s36  
chr               start   end     Lpnp1                          
1                 3000826 3001825 4         0.0     0.0     0.0  
                  3002826 3003825 5         8.0    14.0     7.0  
                  3003826 3004825 2        12.0    18.0    26.0  
                  3005826 3006825 1         0.0     0.0     0.0  
                  3006826 3007825 5         2.0     0.0     1.0  
...                                         ...     ...     ...  
Y_JH584300_random 138923  139922  5         0.0     0.0     0.0  
                  139923  140922  5         0.0     0.0     0.0  
                  140923  141922  2         0.0     0.0     0.0  
                  141923  142922  17        0.0     0.0     0.0  
                  142923  143922  5         0.0     0.0     0.0  

[2213351 rows x 12 columns]

In [16]:
data_merged_1 = pd.concat([all_data[labels[0]], all_data[labels[1]]], axis=1)
data_merged = pd.concat([data_merged_1, all_data[labels[2]]], axis=1)
data_merged

M1_s10  M1_s11  M1_s12  M1_s13  \
chr               start   end     Lpnp1                                   
1                 3000826 3001825 4         0.0     0.0    22.0     0.0   
                  3002826 3003825 5         6.0     0.0     0.0     0.0   
                  3003826 3004825 2         0.0     0.0     4.0    18.0   
                  3005826 3006825 1         0.0     0.0     0.0     0.0   
                  3006826 3007825 5         0.0     0.0     7.0     0.0   
...                                         ...     ...     ...     ...   
Y_JH584300_random 138923  139922  5         0.0     0.0     0.0     0.0   
                  139923  140922  5         0.0     0.0     0.0     0.0   
                  140923  141922  2         0.0     0.0     0.0     0.0   
                  141923  142922  17        0.0     0.0     0.0     0.0   
                  142923  143922  5         0.0     0.0     0.0     0.0   

                                         M1_s24  M1_s25  M1_s26  M1_s27  \
chr               start   end     Lpnp1                                   
1                 3000826 3001825 4         0.0     0.0     0.0     0.0   
                  3002826 3003825 5        12.0     0.0     0.0     0.0   
                  3003826 3004825 2         0.0     4.0     0.0     0.0   
                  3005826 3006825 1         0.0     0.0     0.0     0.0   
                  3006826 3007825 5         0.0     0.0     0.0     0.0   
...                                         ...     ...     ...     ...   
Y_JH584300_random 138923  139922  5         0.0     0.0     0.0     0.0   
                  139923  140922  5         0.0     0.0     0.0     0.0   
                  140923  141922  2         0.0     0.0     0.0     0.0   
                  141923  142922  17        0.0     0.0     0.0     0.0   
                  142923  143922  5         0.0     0.0     0.0     0.0   

                                         M1_s35  M1_s36  ...  M3_s3  M3_s4  \
chr               start   end     Lpnp1                  ...                 
1                 3000826 3001825 4         0.0     0.0  ...    0.0    4.0   
                  3002826 3003825 5        11.0     0.0  ...    4.0    4.0   
                  3003826 3004825 2         0.0     4.0  ...   44.0   22.0   
                  3005826 3006825 1         0.0     0.0  ...    0.0    0.0   
                  3006826 3007825 5         0.0     0.0  ...    1.0    2.0   
...                                         ...     ...  ...    ...    ...   
Y_JH584300_random 138923  139922  5         0.0     0.0  ...    0.0    0.0   
                  139923  140922  5         0.0     0.0  ...    0.0    0.0   
                  140923  141922  2         0.0     0.0  ...    0.0    0.0   
                  141923  142922  17        0.0     0.0  ...    0.0    0.0   
                  142923  143922  5         0.0     0.0  ...    0.0    0.0   

                                         M3_s17  M3_s18  M3_s19  M3_s20  \
chr               start   end     Lpnp1                                   
1                 3000826 3001825 4         0.0     0.0     0.0     0.0   
                  3002826 3003825 5        25.0     8.0    19.0     5.0   
                  3003826 3004825 2        20.0    32.0    20.0    12.0   
                  3005826 3006825 1         0.0     0.0     0.0     0.0   
                  3006826 3007825 5         2.0     7.0     2.0     0.0   
...                                         ...     ...     ...     ...   
Y_JH584300_random 138923  139922  5         0.0     0.0     0.0     0.0   
                  139923  140922  5         0.0     0.0     0.0     0.0   
                  140923  141922  2         0.0     0.0     0.0     0.0   
                  141923  142922  17        0.0     0.0     0.0     0.0   
                  142923  143922  5         0.0     0.0     0.0     0.0   

                                         M3_s33  M3_s34  M3_s35  M3_s36  
chr               start 

In [ ]:
dic_merged={}
dic_merged['M1M2M3_1000bp0ovl_cpg_coverage_table']=data_merged

#save the merged files
outpath=''
saveCSV(dic_merged, outpath, label = '')

In [17]:
#Filter the dataset
data_merged_filterd={}
data_merged_filterd['M1M2M3_1000bp0ovl_cpg_coverage_table_filtered']=data_merged.drop(['M1_s26', 'M1_s37'], axis=1)

In [29]:
#first we need to normalize the data to same coverage in each sections
mean_cov=np.mean(data_merged_filterd['M1M2M3_1000bp0ovl_cpg_coverage_table_filtered'].sum())
data_normal = normalizeTo(data_merged_filterd, factor=mean_cov)
data_normal['M1M2M3_1000bp0ovl_cpg_coverage_table_filtered']

M1_s10  M1_s11     M1_s12  \
chr               start   end     Lpnp1                                 
1                 3000826 3001825 4       0.000000     0.0  26.426519   
                  3002826 3003825 5      11.523931     0.0   0.000000   
                  3003826 3004825 2       0.000000     0.0   4.804822   
                  3005826 3006825 1       0.000000     0.0   0.000000   
                  3006826 3007825 5       0.000000     0.0   8.408438   
...                                            ...     ...        ...   
Y_JH584300_random 138923  139922  5       0.000000     0.0   0.000000   
                  139923  140922  5       0.000000     0.0   0.000000   
                  140923  141922  2       0.000000     0.0   0.000000   
                  141923  142922  17      0.000000     0.0   0.000000   
                  142923  143922  5       0.000000     0.0   0.000000   

                                            M1_s13     M1_s24    M1_s25  \
chr               start   end     Lpnp1                                   
1                 3000826 3001825 4       0.000000   0.000000  0.000000   
                  3002826 3003825 5       0.000000  11.497248  0.000000   
                  3003826 3004825 2      17.809967   0.000000  3.604351   
                  3005826 3006825 1       0.000000   0.000000  0.000000   
                  3006826 3007825 5       0.000000   0.000000  0.000000   
...                                            ...        ...       ...   
Y_JH584300_random 138923  139922  5       0.000000   0.000000  0.000000   
                  139923  140922  5       0.000000   0.000000  0.000000   
                  140923  141922  2       0.000000   0.000000  0.000000   
                  141923  142922  17      0.000000   0.000000  0.000000   
                  142923  143922  5       0.000000   0.000000  0.000000   

                                         M1_s27     M1_s35    M1_s36  M1_s38  \
chr               start   end     Lpnp1                                        
1                 3000826 3001825 4         0.0   0.000000  0.000000     0.0   
                  3002826 3003825 5         0.0  11.964748  0.000000     0.0   
                  3003826 3004825 2         0.0   0.000000  4.350931     0.0   
                  3005826 3006825 1         0.0   0.000000  0.000000     0.0   
                  3006826 3007825 5         0.0   0.000000  0.000000     0.0   
...                                         ...        ...       ...     ...   
Y_JH584300_random 138923  139922  5         0.0   0.000000  0.000000     0.0   
                  139923  140922  5         0.0   0.000000  0.000000     0.0   
                  140923  141922  2         0.0   0.000000  0.000000     0.0   
                  141923  142922  17        0.0   0.000000  0.000000     0.0   
                  142923  143922  5         0.0   0.000000  0.000000     0.0   

                                         ...      M3_s3      M3_s4     M3_s17  \
chr               start   end     Lpnp1  ...                                    
1                 3000826 3001825 4      ...   0.000000   3.786343   0.000000   
                  3002826 3003825 5      ...   3.364549   3.786343  20.855073   
                  3003826 3004825 2      ...  37.010039  20.824888  16.684058   
                  3005826 3006825 1      ...   0.000000   0.000000   0.000000   
                  3006826 3007825 5      ...   0.841137   1.893172   1.668406   
...                                      ...        ...        ...        ...   
Y_JH584300_random 138923  139922  5      ...   0.000000   0.000000   0.000000   
                  139923  140922  5      ...   0.000000   0.000000   0.000000   
                  140923  141922  2      ...   0.000000   0.000000   0.000000   
                  141923  142922  17     ...   0.000000   0.000000   0.000000   
                  142923  143922  5      ...   0.000000   0.000000   0.000000   

                       

In [27]:
dic_merged={}
dic_merged['M1M2M3_1000bp0ovl_cpg_coverage_table_filtered_normal']=data_normal['M1M2M3_1000bp0ovl_cpg_coverage_table_filtered']

#save the merged files
outpath=''
saveCSV(dic_merged, outpath, label = '')


Files saved
